# PPO-GOMDP — Multi-Seed Training on Colab (T4)

Trains the PPO-GOMDP policy for the wildfire governance paper, running several seeds **in parallel** on one T4.

### Runtime required
**Runtime → Change runtime type → T4 GPU + High-RAM.** Verified in Step 1; the notebook stops if the GPU is missing.

### Why the GPU matters here
Measured per-episode split on CPU (grid 100 × 3000 steps):

| Component | Time | Share |
|---|---|---|
| Policy forward (3000 batch-1 passes) | 11.8 s | **71%** |
| PPO update | 2.0 s | 12% |
| Environment (numpy fire sim) | 2.9 s | 17% |

83% is GPU-accelerable, so the T4 is worth using. The remaining 17% is CPU-bound numpy, which is why we also run seeds as parallel processes — that part scales with vCPUs, not the GPU.

### Before you run
This notebook clones from GitHub, so **push your local changes first** — in particular the CUDA support in `ppo_agent.py`, `experiments/11c_train_multiseed.py`, and `requirements-colab.txt`. Cloning an older commit will silently train on CPU.

## Step 1 — Verify the runtime

Fails loudly rather than silently falling back to CPU, which would take ~6 h/seed instead of ~1 h.

In [ ]:
import subprocess, sys, os, multiprocessing

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected.\n"
        "Runtime -> Change runtime type -> Hardware accelerator: T4 GPU, "
        "Runtime shape: High-RAM, then Runtime -> Restart session."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
cpus = multiprocessing.cpu_count()

print(f"torch   : {torch.__version__}  (CUDA {torch.version.cuda})")
print(f"GPU     : {gpu}  |  VRAM {vram:.1f} GB")
print(f"CPU     : {cpus} vCPUs  |  RAM {ram:.1f} GB")

if ram < 30:
    print("\n[WARN] This looks like a standard-RAM runtime. High-RAM is recommended;")
    print("       reduce --workers if you hit out-of-memory errors.")

## Step 2 — Clone the repository

For a **private** repo, replace the URL with a token form:
`https://<GITHUB_TOKEN>@github.com/aliakarma/wildfire-governance-agentic-ai.git`
(use a fine-grained read-only token, and clear the cell output afterwards).

In [ ]:
REPO_URL = "https://github.com/aliakarma/wildfire-governance-agentic-ai.git"
BRANCH   = "main"
REPO_DIR = "/content/wildfire-governance-agentic-ai"

import os, shutil, subprocess
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
    check=True,
)
os.chdir(REPO_DIR)

sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"\ncloned {BRANCH} @ {sha} -> {REPO_DIR}")

# Confirm the clone actually contains the GPU-capable code.
missing = [p for p in ["experiments/11c_train_multiseed.py", "requirements-colab.txt"]
           if not os.path.exists(p)]
has_device = "device" in open("src/wildfire_governance/rl/ppo_agent.py").read()
if missing or not has_device:
    raise SystemExit(
        f"This commit predates the GPU training support (missing: {missing or 'device= in ppo_agent'}).\n"
        "Push your local changes and re-run this cell."
    )
print("GPU training support present.")

## Step 3 — Install dependencies

Uses `requirements-colab.txt`, which omits torch (Colab's CUDA build is kept) and the geospatial stack (only needed for VIIRS preprocessing). Takes well under a minute; installing the full `requirements.txt` would take many minutes and can downgrade torch to a CPU build.

In [ ]:
!pip install -q -r requirements-colab.txt

import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = f"{REPO_DIR}/src:{REPO_DIR}"

import torch
print(f"torch {torch.__version__}  cuda_available={torch.cuda.is_available()}")
assert torch.cuda.is_available(), "pip install downgraded torch to a CPU build — restart runtime and re-run"

## Step 4 — Sanity check

Runs the test suite and a 2-seed smoke train. If either fails, stop here — a long run would only waste GPU hours.

In [ ]:
!python -m pytest tests -q 2>&1 | tail -5
print("\n--- smoke train (2 seeds x 3 episodes) ---")
!python experiments/11c_train_multiseed.py --smoke 2>&1 | grep -v '^{' | tail -12

## Step 5 — Benchmark GPU vs CPU

Times one real episode on each device so the schedule below is based on your actual runtime, not an assumption. Takes ~1 minute.

In [ ]:
import time, numpy as np, torch
from wildfire_governance.rl.gomdp_env import GOMMDPGymEnv
from wildfire_governance.rl.ppo_agent import PPOGOMDPAgent
from wildfire_governance.simulation.grid_environment import EnvironmentConfig

def time_episode(device, grid=100, steps=3000, n_uavs=20):
    env = GOMMDPGymEnv(config=EnvironmentConfig(grid_size=grid, n_timesteps=steps),
                       n_uavs=n_uavs, enable_governance=True)
    agent = PPOGOMDPAgent(grid_size=grid, n_uavs=n_uavs, device=device)
    obs, _ = env.reset(seed=0)
    O, A, R, D = [], [], [], []
    t0 = time.time(); done = False
    while not done:
        ad = agent.select_actions(obs, env._fleet)
        arr = np.array([ad.get(i, 0) for i in range(n_uavs)])
        nobs, r, term, trunc, _ = env.step(arr)
        O.append(obs.copy()); A.append(ad); R.append(float(r)); D.append(term or trunc)
        obs = nobs; done = term or trunc
    roll = time.time() - t0
    t0 = time.time(); agent.update(O, A, R, D); upd = time.time() - t0
    return roll, upd

results = {}
for dev in ("cpu", "cuda"):
    roll, upd = time_episode(dev)
    results[dev] = roll + upd
    print(f"{dev:5s}: rollout {roll:6.2f}s + update {upd:5.2f}s = {roll+upd:6.2f}s/episode")

SEC_PER_EPISODE = results["cuda"]
print(f"\nGPU speedup: {results['cpu'] / results['cuda']:.2f}x")

## Step 6 — Configure the run

`N_SEEDS` is how many independent policies to train (the paper's validation curve is a mean ± SD across seeds). `WORKERS` is how many run concurrently.

**Sizing note.** VRAM is not the limit — each worker needs roughly a 500 MB CUDA context plus ~180 MB of model, optimiser and observation tensors, so even 8 workers use under 6 GB of the T4's 15 GB. The real limits are vCPU count (the 17% numpy portion) and GPU contention between workers. `min(N_SEEDS, vCPUs, 6)` is a safe default; raise it if the GPU shows low utilisation in `nvidia-smi`.

In [ ]:
import multiprocessing

N_SEEDS  = 5      # independent policies; paper's curve uses 5 held-out seeds
EPISODES = 1000   # per seed
WORKERS  = min(N_SEEDS, multiprocessing.cpu_count(), 6)

waves = -(-N_SEEDS // WORKERS)  # ceiling division
est_h = SEC_PER_EPISODE * EPISODES * waves / 3600

print(f"seeds={N_SEEDS}  episodes={EPISODES}  workers={WORKERS}")
print(f"~{SEC_PER_EPISODE:.2f}s/episode  x {EPISODES} episodes x {waves} wave(s)")
print(f"estimated wall-clock: {est_h:.2f} h")
if est_h > 10:
    print("\n[WARN] Exceeds a typical Colab session. Reduce EPISODES or N_SEEDS,")
    print("       or enable the Drive checkpointing in Step 7 so a disconnect is recoverable.")

## Step 7 — (Recommended) Persist results to Google Drive

Colab disconnects lose `/content`. Writing results to Drive means a dropped session costs only the episodes since the last write — the trainer appends its learning curve after **every** episode, so nothing else is lost.

Skip this cell if you'd rather keep everything local and download at the end.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/gomdp_training/multiseed"
else:
    OUTDIR = f"{REPO_DIR}/results/runs/multiseed_colab"

import os
os.makedirs(OUTDIR, exist_ok=True)
print(f"results -> {OUTDIR}")

## Step 8 — Train

Launches in the background so the next cell can show live progress. Each seed appends to its own `ppo_learning_curve.csv` every episode and rewrites `status.json`, so progress is always readable and an interrupted run still leaves usable history.

In [ ]:
import subprocess, os

LOG = "/content/train.log"
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src:{REPO_DIR}")
cmd = [
    "python", "experiments/11c_train_multiseed.py",
    "--seeds", str(N_SEEDS), "--episodes", str(EPISODES),
    "--workers", str(WORKERS), "--device", "cuda", "--outdir", OUTDIR,
]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env,
                        stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print(f"launched pid={proc.pid}; log -> {LOG}")

## Step 9 — Monitor

Re-runnable at any time. Polls each seed's `status.json`. Interrupting this cell does **not** stop training.

In [ ]:
import json, time, glob, os
from IPython.display import clear_output

POLL_S = 30
try:
    while True:
        clear_output(wait=True)
        stats = []
        for p in sorted(glob.glob(f"{OUTDIR}/seed_*/status.json")):
            try:
                stats.append(json.load(open(p)))
            except (json.JSONDecodeError, OSError):
                pass  # mid-write; skip this tick
        if not stats:
            print("waiting for first episode...")
        else:
            print(f"{'seed':>5} {'episode':>12} {'pct':>7} {'best_rw':>10} {'ETA':>10}")
            for s in stats:
                eta = f"{s['eta_s']/60:.0f} min" if s["eta_s"] < 5400 else f"{s['eta_s']/3600:.1f} h"
                print(f"{s['seed']:>5} {s['episode']:>6}/{s['n_episodes']:<5} "
                      f"{s['pct']:>6.1f}% {s['best_reward']:>10.1f} {eta:>10}")
            print(f"\noverall: {sum(x['pct'] for x in stats)/len(stats):.1f}%")
        if proc.poll() is not None:
            print(f"\nTRAINING FINISHED (exit {proc.returncode})")
            break
        time.sleep(POLL_S)
except KeyboardInterrupt:
    print("\n(monitor stopped; training continues in background)")

## Step 10 — Learning curve

Plots validation $L_d$ (mean ± SD across seeds), the figure the paper's training section needs. Reads the aggregate if training finished, otherwise builds it from whatever episodes exist so far.

In [ ]:
import pandas as pd, numpy as np, glob
import matplotlib.pyplot as plt

frames = []
for p in sorted(glob.glob(f"{OUTDIR}/seed_*/ppo_learning_curve.csv")):
    df = pd.read_csv(p)
    df["seed"] = int(p.split("seed_")[1].split("/")[0])
    frames.append(df)

allc = pd.concat(frames, ignore_index=True)
agg = allc.groupby("episode").agg(
    ld_mean=("ld", "mean"), ld_std=("ld", "std"),
    reward_mean=("reward", "mean"),
    compliance_mean=("compliance", "mean"),
).reset_index()

# Smooth for readability; the CSV keeps the raw per-episode values.
W = max(1, len(agg) // 50)
sm = agg.rolling(W, min_periods=1).mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(sm["episode"], sm["ld_mean"], color="tab:blue")
ax[0].fill_between(sm["episode"], sm["ld_mean"] - sm["ld_std"],
                   sm["ld_mean"] + sm["ld_std"], alpha=0.2, color="tab:blue")
ax[0].set_xlabel("Training episode"); ax[0].set_ylabel("Validation $L_d$ (steps)")
ax[0].set_title(f"PPO-GOMDP learning curve ({allc['seed'].nunique()} seeds)")
ax[0].grid(alpha=0.3)

ax[1].plot(sm["episode"], sm["reward_mean"], color="tab:orange")
ax[1].set_xlabel("Training episode"); ax[1].set_ylabel("Episode reward")
ax[1].set_title("Reward"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

tail = agg.tail(50)
print(f"episodes completed : {len(agg)}")
print(f"final L_d (last 50): {tail['ld_mean'].mean():.2f} +/- {tail['ld_std'].mean():.2f}")
print(f"governance compliance: {100*agg['compliance_mean'].mean():.1f}%   <- Theorem 1")
agg.to_csv(f"{OUTDIR}/learning_curve_aggregate.csv", index=False)

## Step 11 — Export

Packages checkpoints, curves and summary for download. Copy `best_checkpoint.pt` into `src/wildfire_governance/rl/checkpoints/ppo_gomdp_best.pt` locally to use it for evaluation.

In [ ]:
import shutil, json, os, glob

summary_path = f"{OUTDIR}/summary.json"
if os.path.exists(summary_path):
    print(json.dumps(json.load(open(summary_path)), indent=2)[:1200])

archive = shutil.make_archive("/content/gomdp_training_results", "zip", OUTDIR)
print(f"\n{archive}  ({os.path.getsize(archive)/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"(auto-download unavailable: {exc}; results are in {OUTDIR})")